# RecursiveCharacterTextSplitter 실습

`CharacterTextSplitter`에 이어, 여러 구분자를 순서대로 시도하면서 최대한 `chunk_size`에 맞춰 텍스트를 나누는 `RecursiveCharacterTextSplitter`를 실습한다. 같은 용어집 텍스트(`data/appendix-keywords.txt`)를 사용하되, 이번에는 `chunk_overlap`을 0이 아닌 값으로 지정해서 조각 사이에 내용이 겹치는 것도 함께 확인한다. 텍스트 분할만 다루는 노트북이라 LLM API 호출이나 `.env` 로딩은 필요 없다.

In [2]:
# 용어집 텍스트 파일을 UTF-8 인코딩으로 읽어온다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

## 1. 원본 텍스트 미리보기

읽어온 텍스트의 앞부분 500자를 출력해서 내용을 확인한다. "Semantic Search", "Embedding", "Token" 순으로 용어가 정의·예시·연관키워드 형식으로 정리되어 있고, 용어 사이는 빈 줄(`\n\n`)로 구분되어 있다.

In [3]:
print(file[:500])

Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어


## 2. RecursiveCharacterTextSplitter 임포트

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 3. 분할기 설정

`RecursiveCharacterTextSplitter`는 기본적으로 `["\n\n", "\n", " ", ""]` 순서의 구분자 목록을 차례로 시도하면서, 가능한 한 의미 단위(문단 → 줄 → 단어)를 지키는 선에서 `chunk_size`에 맞게 나눈다.

- `chunk_size=250` : 조각 하나의 최대 글자 수.
- `chunk_overlap=50` : 조각들 사이에 50자만큼 내용을 겹쳐서, 조각 경계에서 문맥이 끊기는 것을 줄인다.
- `length_function=len` : 글자 수 계산에 사용할 함수.
- `is_separator_regex=False` : 구분자를 정규식이 아닌 일반 문자열로 취급.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

## 4. create_documents()로 분할하고 겹침(overlap) 확인

첫 번째 조각(`texts[0]`)은 "Semantic Search" 항목 전체를 담고 다음 항목 제목("Embedding")까지 포함한 뒤 끝난다. 두 번째 조각(`texts[1]`)을 보면 "Embedding"부터 다시 시작하는데, 이게 바로 `chunk_overlap=50`의 효과다 — 두 조각이 "Embedding"이라는 경계 지점을 공유하면서 이어지기 때문에, 조각만 따로 읽어도 문맥이 뚝 끊기지 않는다.

In [9]:
texts = text_splitter.create_documents([file])
print(texts[0])
print("===" * 20)
print(texts[1])

page_content='Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding'
page_content='Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token'


## 5. split_text(): 순수 문자열로 확인

`CharacterTextSplitter`와 마찬가지로 `split_text()`는 `Document`로 감싸지 않고 순수 문자열 리스트를 반환한다. 앞의 두 조각(`[:2]`)을 출력해보면 4번의 `texts[0]`, `texts[1]`의 `page_content`와 동일한 내용(겹치는 "Embedding" 부분 포함)임을 확인할 수 있다.

In [10]:
text_splitter.split_text(file)[:2]

['Semantic Search\n\n정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.\n예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.\n연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝\n\nEmbedding',
 'Embedding\n\n정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.\n예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.\n연관키워드: 자연어 처리, 벡터화, 딥러닝\n\nToken']

## 정리

| 분할기 | 나누는 기준 | chunk_size 초과 시 |
|---|---|---|
| `CharacterTextSplitter` | 지정한 구분자 하나 | 더 잘게 쪼개지 않고 그대로 둠 |
| `RecursiveCharacterTextSplitter` | 여러 구분자를 순서대로 시도(`\n\n` → `\n` → 공백 → 글자) | 다음 구분자로 더 잘게 나눠 최대한 chunk_size에 맞춤 |

`chunk_overlap`을 0보다 크게 주면 인접한 조각들이 경계 부분을 일부 공유하게 되어, 조각 하나만 떼어봐도 문맥이 급격히 끊기지 않는다. 다만 그만큼 전체 조각 수와 저장 용량은 늘어난다는 트레이드오프가 있다.